# Lab Instructions

In the lab, you're presented a task such as building a dataset, training a model, or writing a training loop, and we'll provide the code structured in such a way that you can fill in the blanks in the code using the knowledge you acquired in the chapters that precede the lab. You should be able to find appropriate snippets of code in the course content that work well in the lab with minor or no adjustments.

The blanks in the code are indicated by ellipsis (`...`) and comments (`# write your code here`).

In some cases, we'll provide you partial code to ensure the right variables are populated and any code that follows it runs accordingly.

```python
# write your code here
x = ...
```

The solution should be a single statement that replaces the ellipsis, such as:

```python
# write your code here
x = [0, 1, 2]
```

In some other cases, when there is no new variable being created, the blanks are shown like in the example below: 

```python
# write your code here
...
```

Although we're showing you only a single ellipsis (`...`), you may have to write more than one line of code to complete the step, such as:

```python
# write your code here
for i, xi in enumerate(x):
    x[i] = xi * 2
```

## Installation Notes

To run this notebook on Google Colab, install `torchmetrics`:

In [ ]:
!pip install torchmetrics

The dataset functions implemented in Chapter 12 can be easily imported from a set of helper functions we're making available for your convenience. You can download it from the following link:

`https://raw.githubusercontent.com/lftraining/LFD273-code/main/object_detection_functions.py`

In Google Colab, you can run the following command to download the file:

In [ ]:
!wget https://raw.githubusercontent.com/lftraining/LFD273-code/main/object_detection_functions.py

Once the file is downloaded, you only need to import the required functions:

In [ ]:
from object_detection_functions import *

## 13.4 Lab 5B: Overview

In Lab 5A, you prepared the Oxford-IIIT Pet data for Torchvision detection models. In this second part, you will fine-tune and evaluate a pretrained Faster R-CNN detector.

You will replace its prediction head, configure optimization and learning-rate scheduling, train the detector, save and reload its weights, inspect predictions, calculate IoU, and evaluate the validation set using mean Average Precision (mAP).

## 13.4.1 Recap

Lab 5B depends on the completed Lab 5A data pipeline. To make this notebook runnable in a fresh Colab session, the finalized Lab 5A setup is reproduced below as **provided code**.

In [ ]:
import json
import torch
import tarfile
import pandas as pd
from pathlib import Path
from datasets import Dataset, Features, Image as HFImage, Sequence, Value, ClassLabel
from torchvision.io import read_image, ImageReadMode
from torchvision.ops import masks_to_boxes, box_area, box_convert
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from object_detection_functions import xml_to_csv, build_object_detection_dataset, build_detection_transform

First, we download images and annotations for the Oxford-IIIT Pet Dataset, and extract them locally:

In [ ]:
!wget https://thor.robots.ox.ac.uk/~vgg/data/pets/annotations.tar.gz
!wget https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
# Alternative mirror
#!wget https://datasets.cms.waikato.ac.nz/ufdl/data/oxford-pets/annotations.tar.gz
#!wget https://datasets.cms.waikato.ac.nz/ufdl/data/oxford-pets/images.tar.gz

In [ ]:
# Open and extract the archive automatically detecting compression (mode='r:*')
with tarfile.open('annotations.tar.gz', 'r:*') as tar:
    tar.extractall()
print("Annotations extracted successfully!")
with tarfile.open('images.tar.gz', 'r:*') as tar:
    tar.extractall()
print("Images extracted successfully!")

Then, we convert the XML annotations into a dataframe using the `xml_to_csv()` helper function, and built a dictionary of categories:

In [ ]:
xml_df = xml_to_csv("annotations/xmls")

trainval_df = pd.read_csv('annotations/trainval.txt', sep=' ', header=None, names=['filename', 'class_id', 'species', 'breed_id'])
trainval_df['category'] = trainval_df['filename'].apply(lambda v: ' '.join([w.capitalize()
                                                                            for w in v.split('_')[:-1]]))

trainval_df['filename'] = trainval_df['filename'].apply(lambda v: f'{v}.jpg')
annotations_df = trainval_df.merge(xml_df, how='left', on='filename')

colnames = ['filename', 'label', 'category', 'width', 'height', 'xmin', 'ymin', 'xmax', 'ymax']
annotations_df = annotations_df.rename(columns={'class_id': 'label'})[colnames]

id2label = dict(annotations_df[['label', 'category']].drop_duplicates().values)

annotations_df = annotations_df.dropna().copy()
annotations_df["box_width"] = (annotations_df["xmax"] - annotations_df["xmin"])
annotations_df["box_height"] = (annotations_df["ymax"] - annotations_df["ymin"])
# make classes zero-based for our HF-style dataset
annotations_df["label"] = annotations_df["label"] - 1

Next, we use the annotations to write the `metadata.jsonl` file:

In [ ]:
records = []
for filename, group in annotations_df.groupby("filename", sort=False):
    objects = {
        "bbox": [[float(row.xmin), float(row.ymin), float(row.box_width), float(row.box_height)]
                for row in group.itertuples()],
        "category": [int(label) for label in group["label"]],
    }

    record = {
        "file_name": filename,
        "width": int(group.iloc[0]["width"]),
        "height": int(group.iloc[0]["height"]),
        "objects": objects,
        "label": int(group.iloc[0]["label"])
    }
    records.append(record)

metadata_file = Path("metadata.jsonl")
with metadata_file.open("w", encoding="utf-8") as f:
    f.writelines([json.dumps(record) + '\n' for record in records])

We can then use the `build_object_detection_dataset()` helper function to load the images and their correspondinng metadata. We also cast the label column as a `ClassLabel` so it can be used to split the dataset in a stratified fashion:

In [ ]:
image_folder = './images'
pets_detection = build_object_detection_dataset(image_folder, metadata_file=metadata_file)
pets_detection = pets_detection.cast_column("label", ClassLabel(num_classes=37))
datasets = pets_detection.shuffle(seed=42).train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
datasets["validation"] = datasets.pop("test")

Preprocessing images and performing data augmentation is a big part of training a model, so we create functions to perform transformations that will be lazily applied to the dataset splits:

In [ ]:
train_transforms = v2.Compose([v2.RandomHorizontalFlip(p=0.5), v2.SanitizeBoundingBoxes(), v2.ToDtype(torch.float32, scale=True)])
val_transforms = v2.Compose([v2.SanitizeBoundingBoxes(), v2.ToDtype(torch.float32, scale=True)])

train_fn = build_detection_transform(train_transforms)
val_fn = build_detection_transform(val_transforms)

datasets['train'] = datasets['train'].with_transform(train_fn)
datasets['validation'] = datasets['validation'].with_transform(val_fn)

Once the datasets are ready, we create data loaders using the collate function so we can load mini-batches of data, one at a time.

In [ ]:
def collate_fn(batch):
    images = [item["image"] for item in batch]
    targets = [item["target"] for item in batch]
    return images, targets
    
dataloaders = {}
dataloaders['train'] = DataLoader(datasets['train'], batch_size=4, shuffle=True, collate_fn=collate_fn)
dataloaders['validation'] = DataLoader(datasets['validation'], batch_size=4, shuffle=False, collate_fn=collate_fn)

### 13.4.2 Load a Pretrained Faster R-CNN

Start from Faster R-CNN with a MobileNetV3-Large backbone and FPN, using the default pretrained weights.

In [ ]:
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights

# write your code here
weights = ...
model = ...

Inspect the detector's built-in image transformation component.

In [ ]:
model.transform

### 13.4.3 Replace the Head (Box Predictor)

First, inspect the original box predictor, you will see that it was trained for a different set of classes and must be replaced.

In [ ]:
model.roi_heads.box_predictor

 You already know the number of classes - but don't forget another one for the negative case, that is, whenever there's no object in the image. This class (for the background, if you will) is usually assigned the zero index.

In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# write your code here
in_features = ...
model.roi_heads.box_predictor = ...

Once the model has been updated, we must move it to the appropriate device.

In [ ]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
model = model.to(device)

### 13.4.4 Configure the Optimizer and Learning-Rate Schedulers

For the optimizer, we'll use SGD with momentum and weight decay, optionally combined with schedulers, as suggested below:
- `LinearLR` warms up the learning rate during the first epoch;
- `StepLR` reduces it by a factor of ten every three epochs.

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, LinearLR

# write your code here
optimizer = ...

# Decreases the learning rate by 10x every 3 epochs
# write your code here
lr_scheduler = ...

# Warms-up the learning rate from zero to 0.005 over one epoch
warmup_factor = 1.0 / 1000
warmup_iters = min(1000, len(dataloaders['train']) - 1)

# write your code here
lr_scheduler2 = ...

### 13.4.5 Fine-Tune the Detector

It is time to write a real training loop now! You can use the dummy loop as a template and build on top of it, once you're happy with your schedulers. Don't forget to send every tensor, individually, to the same device as the model. Also, keep in mind that the model returns a dictionary with many separate losses. It is your job to sum them all up to compute gradients based on the total.

In [ ]:
num_epochs = 5

model.to(device)

for epoch in range(num_epochs):
    print(f"Epoch {epoch}\n")
    for i, (images, targets) in enumerate(dataloaders['train']):
        # Send images and targets to device
        # write your code here
        ...

        # Set your model's mode
        # write your code here
        ...

        # Call the model to get a loss dict back
        # write your code here
        ...

        if not (i % 50):
            print([(k, v.item()) for k, v in loss_dict.items()])

        # You have many losses in the dict, but you can only
        # call backward one a single value, so you must
        # add them up
        # write your code here
        ...

        optimizer.step()
        optimizer.zero_grad()

        if epoch == 0:
            lr_scheduler2.step()

    lr_scheduler.step()

### 13.4.6 Save and Reload the Weights

Once training is finished, we can save the model to disk.

In [ ]:
torch.save(model.state_dict(), 'mobilenet_v3_large_pets.pth')

And then reload the saved parameters into the same model architecture, as we would do if we were about to deploy the model to run inference on it.

In [ ]:
weights = torch.load('mobilenet_v3_large_pets.pth', map_location=device)
model.load_state_dict(weights)

### 13.4.7 Run Inference

Let's try the model out! First, select one validation example containing both the image and its reference target.

In [ ]:
sample = datasets['validation'][0]

image = sample["image"]
target = sample["target"]

Then, set the model to the appropriate mode and run inference under `torch.inference_mode()`. The detector returns one prediction dictionary per image.

In [ ]:
# write your code here
# set the model's mode
...

with torch.inference_mode():
    # write your code here
    prediction = ...
    prediction = {key: value.cpu() for key, value in prediction.items()}

prediction, target['boxes']

How many boxes did the model predict? In most cases, there is only one actual box in each of our images. Don't forget to check the associated scores.

#### 13.4.7.1 Filter Predictions

Low-confidence boxes can be filtered out. Assign a threshold and filter them out.

In [ ]:
# write your code here
threshold = ...
keep = ...
boxes = prediction["boxes"][keep]
labels = prediction["labels"][keep]
scores = prediction["scores"][keep]

#### 13.4.7.2 Visualize Predictions

Let's compare the actual and predicted boxes. Draw the boxes over the image, the former in green and the latter in red using `draw_bounding_boxes()`.

In [ ]:
from torchvision.utils import draw_bounding_boxes
from torchvision.transforms import ToPILImage

# write your code here
image_with_boxes = ...

ToPILImage()(image_with_boxes)

Does the predicted box match the actual one? How would you assess it in a quantitative way? Can you come up with an appropriate metric? That's the topic of the next chapter!